In [22]:
#code for tracing particles back to SBZ draft (python version 3.10.9) (optimized with numpy.where)
#(1) row each row in pinfo:
#(2) if pinfo[row,3]>pinfo[row,4], then find that particle in data and trace back to when w<0.5
#(3) then report max value of convergence within 2kmx2km box

import numpy as np
import matplotlib.pyplot as plt
import xarray as xr
import os; import time
from multiprocessing import Process, Manager, Lock
if os.path.exists('/mnt/lustre/koa/koastore/torri_group/air/parcel_tracking.nc'): os.remove('/mnt/lustre/koa/koastore/torri_group/air/parcel_tracking.nc')

start_time = time.time();

data=xr.open_dataset('/mnt/lustre/koa/koastore/torri_group/air/cm1r20.3/run/cm1out_test6.nc') #***
parcel=xr.open_dataset('/mnt/lustre/koa/koastore/torri_group/air/cm1r20.3/run/cm1out_pdata_test6.nc') #***
times=data['time'].values/(1e9 * 60); times=times.astype(float);
parcel=parcel.isel(time=np.arange(0, len(parcel['time']), int(times[1]))) 
whereCL=xr.open_dataset('/mnt/lustre/koa/koastore/torri_group/air/whereCL_6.nc') #***

#initialization
################################################################################################################################################################################################################

thresh=1e-1/1000
CLmaxheight=750
ascend_thresh=0e-1/1000
ascend_percent=1.00 #0.00 0.25 #0.50 #0.75
ascend=np.argmax(times >= 5) #founds how many timesteps is 5 minutes simulation time (always 1 if data timestep > 5)

starttime= 525 #85 #525 #***

################################################################################################################################################################################################################

def get_3dtime_data(data,varname,tlev):
    cloud_var=data[varname].isel(time=tlev).values
    return cloud_var

def parcel_info(parcel,data,t,visited_edit): #info for all parcels at single time
    x=parcel['x'].isel(time=t).values/1000; xf=data['xf'].values; which_x=np.searchsorted(xf,x)-1; which_x[which_x==-1]=0 #finds which x layer parcel in
    y=parcel['y'].isel(time=t).values/1000; yf=data['yf'].values; which_y=np.searchsorted(yf,y)-1; which_y[which_y==-1]=0 #finds which y layer parcel in
    z=parcel['z'].isel(time=t).values/1000; #zf=data['zf'].values; which_z=np.searchsorted(zf,z)-1; which_z[which_z==-1]=0

    looprange=[ind for ind in range(len(parcel['xh'])) if ind not in visited_edit]
    pinfo=np.zeros((len(parcel['xh']),6),dtype='object') #num,x,y,z, LFC, LCL
    for ind in looprange: #parcel num, whichx, whichy, whichz, whichlfc
        #Find LFC levels 
        lfc=data['lfc'].isel(time=t,xh=which_x[ind],yh=which_y[ind]).values/1000;
        lcl=data['lcl'].isel(time=t,xh=which_x[ind],yh=which_y[ind]).values/1000;
        if lfc<0:
            # print(f'no lfc value available for parcel {num} at time {t}. using neighboring lfc average.')
            xrange=np.mod(np.arange(which_x[ind]-10,which_x[ind]+10),len(data['xh'])); yrange=np.mod(np.arange(which_y[ind]-10,which_y[ind]+10),len(data['yh'])); 
            lfc=data['lfc'].isel(time=t,xh=xrange,yh=yrange).mean().values/1000
        if lcl<0:
            # print(f'no lcl value available at time {t} for parcel {num}. using neighboring lfc.')
            xrange=np.mod(np.arange(which_x[ind]-10,which_x[ind]+10),len(data['xh'])); yrange=np.mod(np.arange(which_y[ind]-10,which_y[ind]+10),len(data['yh'])); 
            lcl=data['lcl'].isel(time=t,xh=xrange,yh=yrange).mean().values/1000   
        
        pinfo[ind,0]=ind; pinfo[ind,1]=which_x[ind]; pinfo[ind,2]=which_y[ind]; pinfo[ind,3]=z[ind]; pinfo[ind,4]=lfc
        pinfo[ind,5]=lcl #marks tracer number at individual columns
    if len(visited_edit)>0: pinfo=np.delete(pinfo, visited_edit, axis=0)
    return(pinfo)

def parcel_info_back(parcel,data,back_t,visited_back_edit): #info for single parcels for current back_time
    looprange=[ind for ind in range(len(parcel['xh'])) if ind not in visited_back_edit]
    pinfo_s_out=np.zeros((len(looprange),10),dtype='object') #num,x,y,z, LFC, LCL
    for ind,num in enumerate(looprange):  
        x=parcel['x'].isel(time=back_t,xh=num).values/1000; xf=data['xf'].values; which_x=np.searchsorted(xf,x)-1; 
        if which_x==-1: which_x=0 #finds which x layer parcel in
        y=parcel['y'].isel(time=back_t,xh=num).values/1000; yf=data['yf'].values; which_y=np.searchsorted(yf,y)-1; 
        if which_y==-1: which_y=0 #finds which y layer parcel in
        z=parcel['z'].isel(time=back_t,xh=num).values/1000; zf=data['zf'].values; which_z=np.searchsorted(zf,z)-1; 
        if which_z==-1: which_z=0
        u=parcel['u'].isel(time=back_t,xh=num).values/1000;
        v=parcel['v'].isel(time=back_t,xh=num).values/1000;
        w=parcel['w'].isel(time=back_t,xh=num).values/1000;
        
        #marks parcel num,which_x,which_y,z,u,v,w,x, and which_z at individual columns
        pinfo_s_out[ind,0]=num; pinfo_s_out[ind,1]=which_x; pinfo_s_out[ind,2]=which_y; pinfo_s_out[ind,3]=z; 
        pinfo_s_out[ind,4]=u; pinfo_s_out[ind,5]=v; pinfo_s_out[ind,6]=w; pinfo_s_out[ind,7]=x; pinfo_s_out[ind,8]=which_z #use z=3 if want to use CL level 3 for all levels
    return(pinfo_s_out)

################################################################################################################################################################################################################

visited=[];visited_edit=[] #stores variables to forgot
out_arr=np.zeros((len(parcel['xh']),6),dtype='object') #num,x,y,z,time
save_arr=np.zeros((len(parcel['xh']),6),dtype='object') #to save forgotten parcels
for t in range(starttime,len(whereCL['time'].values)): 
# for t in range(1086,1086+1): #*test*
    if np.mod(t,1)==0: print(f'\t\t\t\t\t\t\t\t\t\t\t\tcurrent time step: {t}\n')
    
    #remove visited parcels from pinfo and load pinfo
    ###################################################################################################################################
    visited_edit=[ind for ind in visited if ind in pinfo[:, 0]] #parcel number to delete that are stored in pinfo
    if len(visited_edit)>0: visited_edit=[np.where(pinfo[:,0]==ind)[0][0] for ind in visited_edit] #convert parcel num to indices in pinfo
    pinfo=parcel_info(parcel,data,t,visited_edit) #array storing information about all parcels
    ###################################################################################################################################
    
    #chooses info of parcels that are above LFC
    ###################################################################################################################################
    condition1=np.greater_equal(pinfo[:,3],pinfo[:,4])*np.less_equal(pinfo[:,3],pinfo[:,4]+1000/1000)
    condition2=np.greater(pinfo[:,4],0); conditions=condition1*condition2*1
    pinfo=pinfo[np.where(conditions==1)] #parcel info is constrained to LFC condition
    print(f'number of parcels above LFC: {len(pinfo)}\n')
    if len(pinfo)==0: 
        print('break1') #*test
        break #breaks if pinfo is empty
    ###################################################################################################################################
    
    #choose parcels that satisfy ascend condition
    ###################################################################################################################################
    ascend_range=np.arange(t+1,t+1+ascend,1); ascend_range=ascend_range[ascend_range<len(times)]
    if np.any(ascend_range): 
        ascend_array=parcel['w'].isel(time=ascend_range,xh=pinfo[:,0].astype(int)).values/1000; 
        def ascend_condition(column):
            return np.where(np.sum(column >= ascend_thresh) / len(column) >= ascend_percent, 1, 0)
        conditions_false=np.where(np.apply_along_axis(ascend_condition, 0, ascend_array)==0)
        conditions=np.where(np.apply_along_axis(ascend_condition, 0, ascend_array)==1)
        print(f'{len(pinfo[conditions_false][:,0])} parcels hit LFC but not ascending enough after.\n')  
        pinfo=pinfo[np.where(np.apply_along_axis(ascend_condition, 0, ascend_array)==1)] #parcel info is constrained to ascend condition
    if len(pinfo)==0: 
        print('break2') #*test
        continue #breaks if pinfo is empty
    ###################################################################################################################################
    
    ###################################################################################################################################
    #runs through leftover parcels and checks if w small under 750 m and near CL
    #trace back in time until w<0.05 or reach t=0 without w>0.05
    visited_back=visited.copy() #stores variables to forgot parcel for all back_times
    for back_t in np.arange(t-1, starttime-1, -1): #counts from current time down to starttime  
        print(f'\t\t\tback time: {back_t}')
        
        #remove visited parcels from pinfo_s and loading parcel info for back time
        ###################################################################################################################################
        visited_back_edit=[ind for ind in visited_back if ind in pinfo[:, 0]] #only keep parcels to delete that are stored in pinfo_s
        if len(visited_back_edit)>0: visited_back_edit=[np.where(pinfo[:,0]==ind)[0][0] for ind in visited_back_edit] #convert parcel num to indices in pinfo_s
        pinfo_s=parcel_info_back(parcel,data,back_t,visited_back_edit) #gets info of current parcel
        print(f'pinfo_s size: {len(pinfo_s)}\n')
        if len(pinfo_s)==0:  
            print('break3') #*test
            break
        ###################################################################################################################################
        
        #forgets parcels that cross boundary in x or z (x is radiative, z non-periodic)
        ###################################################################################################################################
        condition1=np.greater(pinfo_s[:,7]+pinfo_s[:,4],np.max(data['xf'].values))
        condition2=np.greater(pinfo_s[:,3]+pinfo_s[:,6],np.max(data['zf'].values))
        conditions=np.where(condition1+condition2*1==1) #finds incidies in pinfo_s where conditions are true
        if conditions[0].size>0:
            print(f'parcel {pinfo_s[:,0][conditions]} crossed x,y,or,z boundary, will forget\n')
            #stores parcel info to save array
            save_arr[pinfo_s[:,0][conditions].astype(int),:4]=pinfo_s[conditions][:,:4]
            save_arr[pinfo_s[:,0][conditions].astype(int),4]=back_t; save_arr[pinfo_s[:,0][conditions].astype(int),5]=t  
            visited.extend(pinfo_s[:,0][conditions]) #delete for future times
            visited_back.extend(pinfo_s[:,0][conditions]) #delete for future back times
            pinfo_s=np.delete(pinfo_s, np.where(np.isin(pinfo_s[:, 0], pinfo_s[:, 0][conditions])), axis=0) #delete for this back time
        if len(pinfo_s)==0:  
            print('break4') #*test
            break
        ###################################################################################################################################

        #forgets parcels where w<thresh but above CLmaxheight       
        ###################################################################################################################################
        conditions=np.where(np.less(pinfo_s[:,6],thresh)*np.greater_equal(pinfo_s[:,3],CLmaxheight/1000)*1==1) 
        if conditions[0].size>0:
            print(f'parcel {pinfo_s[:,0][conditions]} w < {thresh*1000} m/s at time but above LCL = {back_t}\n')     
            visited_back.extend(pinfo_s[:,0][conditions]) #forgot for rest of back times for this time
            pinfo_s=np.delete(pinfo_s, np.where(np.isin(pinfo_s[:, 0], pinfo_s[:, 0][conditions])), axis=0) #delete for this back time 
        if len(pinfo_s)==0:  
            print('break5') #*test
            break 
        ###################################################################################################################################
    
        #keeps parcels only where w<thresh and below CLmaxheight
        ###################################################################################################################################
        conditions=np.where(np.less(pinfo_s[:,6],thresh)*np.less(pinfo_s[:,3],CLmaxheight/1000)*1==1) 
        if conditions[0].size>0:
            print(f'parcel {pinfo_s[:,0][conditions]} w < {thresh*1000} m/s at time and below LCL = {back_t}\n')   
            pinfo_s=pinfo_s[conditions] 
        else:
            pinfo_s=np.zeros((0,0),dtype='object')   
            continue
        ###################################################################################################################################
        
        #get the data for the x grid location for the CL
        #checks if the x,y CL indicies match the x,y incidicies of the current parcel
        #if so append to out_array and forget
        ###################################################################################################################################
        for row in range(len(pinfo_s)):
            pinfo_s[row,9]=int(whereCL.isel(time=back_t,z=pinfo_s[row,8],y=pinfo_s[row,2])['maxconv_x'].values) 
        if np.any(pinfo_s[:,9]>0): print(f'Current parcel x locations: {pinfo_s[:,1]}'); print(f'Current parcel CL_x locations: {pinfo_s[:,9]}\n')
            
        def condition(row):
            kms=np.argmax(data['xh'].values-data['xh'][0].values >= 1) #finds how many x grids is 1 km
            return np.isin(pinfo_s[row,9], np.arange(pinfo_s[row,1]-2*kms,pinfo_s[row,1]+3*kms))
        conditions=[]
        for row in range(len(pinfo_s)):
            conditions.append(condition(row))
        conditions=np.where(np.array(conditions)*1==1)

        if conditions[0].size>0:
            print(f'parcel {pinfo_s[:,0][conditions]} is near CL at t = {back_t}. parcel saved.\n')
            #stores parcel info to out_array
            out_arr[pinfo_s[:,0][conditions].astype(int),:4]=pinfo_s[conditions][:,:4]
            out_arr[pinfo_s[:,0][conditions].astype(int),4]=back_t; out_arr[pinfo_s[:,0][conditions].astype(int),5]=t 
            visited.extend(pinfo_s[:,0][conditions]) #append the visited particle number to delete at next time step 
            visited_back.extend(pinfo_s[:,0][conditions]) 
        else:
            visited_back.extend(pinfo_s[:,0]) #delete non-CL parcels for future back times
        ###################################################################################################################################
        # import sys; sys.exit() #*test*
    ###################################################################################################################################
    #storing output and save data
    ds=xr.Dataset({
        'out_arr': (['rows', 'columns'], out_arr.astype(float)),
        'save_arr': (['rows', 'columns'], save_arr.astype(float)),
    })
    ds.to_netcdf('parcel_tracking.nc')
    ###################################################################################################################################
end_time = time.time(); elapsed_time = end_time - start_time; print(f"Total Elapsed Time: {elapsed_time} seconds\n") #(5d30m1000p: 7-8 minutes)
################################################################################################################################################################################################################

												current time step: 1086

number of parcels above LFC: 228

118 parcels hit LFC but not ascending enough after.

			back time: 1085
pinfo_s size: 110

parcel [2 4 28 48 55 57 67 74 99 115 118 127 129 131 185 188 226 229 244 261 265
 292 311 319 329 349 364 377 393 397 411 415 416 425 426 427 432 464 468
 484 485 501 523 539 552 559 588 597 616 620 621 633 635 656 664 673 687
 698 705 710 722 734 737 761 785 789 791 805 819 822 825 832 836 841 847
 854 860 875 878 887 888 893 910 942 946 953 959 967 979 995] w < 0.1 m/s at time but above LCL = 1085

			back time: 1084
pinfo_s size: 20

parcel [82 85 458 479 672 772 932 963] w < 0.1 m/s at time but above LCL = 1084

			back time: 1083
pinfo_s size: 12

parcel [11 277 441 472 541] w < 0.1 m/s at time but above LCL = 1083

			back time: 1082
pinfo_s size: 7

parcel [284 986] w < 0.1 m/s at time but above LCL = 1082

parcel [573 876] w < 0.1 m/s at time and below LCL = 1082

Current parcel x locations: [325 327]
Current parcel CL

In [15]:
##### viewing tracking algorithm results
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr
import os; import time
data=xr.open_dataset('/mnt/lustre/koa/koastore/torri_group/air/cm1r20.3/run/cm1out_test5.nc') #***
parcel=xr.open_dataset('/mnt/lustre/koa/koastore/torri_group/air/cm1r20.3/run/cm1out_pdata_test5.nc') #***
times=data['time'].values/(1e9 * 60); times=times.astype(float);
parcel=parcel.isel(time=np.arange(0, len(parcel['time']), int(times[1]))) 
whereCL=xr.open_dataset('/mnt/lustre/koa/koastore/torri_group/air/whereCL_5.nc') #***

out=xr.open_dataset('parcel_tracking_5opt.nc')['out_arr'].values;out=out.astype(object);out[:, [0,1,2,4,5]] = out[:, [0,1,2,4,5]].astype(int) #***
save=xr.open_dataset('parcel_tracking_5opt.nc')['save_arr'].values;save=save.astype(object);save[:, [0,1,2,4,5]] = save[:, [0,1,2,4,5]].astype(int) #***

out_nz=out[~np.all(out == 0, axis=1)];print('list of first 10 SBZ parcels'); print(out_nz[:15])
save_nz=save[~np.all(save == 0, axis=1)];save_nz=save_nz[np.where(np.unique(save_nz[1:-1,0]))];print('list of first 10 ignored parcels');print(save_nz[:5])
print(f'there are a total of {len(out_nz)} SBZ parcels and {len(save_nz)} forgotten parcels')

list=[]
print('checking if all found SBZ parcels originate from CL')
for ind in range(0,out_nz.shape[0]):
    x=parcel['x'].isel(time=out_nz[ind,4],xh=out_nz[ind,0]).values/1000;xf=data['xf'].values; which_x=np.searchsorted(xf,x)-1 
    z=parcel['z'].isel(time=out_nz[ind,4],xh=out_nz[ind,0]).values/1000; zf=data['zf'].values; which_z=np.searchsorted(zf,z)-1;
    maxconv_x=whereCL['maxconv_x'].isel(time=out_nz[ind,4],z=which_z).values
    list.append(any(np.isin(maxconv_x, np.arange(which_x-2,which_x+3))))
print(list)
list=[]
print('checking if all found SBZ parcels hit LFC')
for ind in range(0,out_nz.shape[0]):
    xloc=parcel['x'].isel(time=out_nz[ind,5],xh=out_nz[ind,0]).values/1000; yloc=parcel['y'].isel(time=out_nz[ind,5],xh=out_nz[ind,0]).values/1000
    xf=data['xf'].values; which_x=np.searchsorted(xf,xloc)-1; yf=data['yf'].values; which_y=np.searchsorted(yf,yloc)-1; 
    z=parcel['z'].isel(time=out_nz[ind,5],xh=out_nz[ind,0]).values/1000
    lfc=data['lfc'].isel(time=out_nz[ind,5],xh=which_x,yh=which_y).values/1000
    list.append(z>=lfc)
print(list)
out_min=np.round(np.min(out_nz[:,3]),3);out_max=np.round(np.max(out_nz[:,3]),3); print(f"CL zlev range is {out_min}:{out_max:.3f} km")
out_mean=np.round(np.mean(out_nz[:,3]),3); print(f'mean CL zlev is {out_mean} km')

list of first 10 SBZ parcels
[[972 314 28 0.5938472290039063 178 179]]
list of first 10 ignored parcels
[]
there are a total of 1 SBZ parcels and 0 forgotten parcels
checking if all found SBZ parcels originate from CL
[True]
checking if all found SBZ parcels hit LFC
[True]
CL zlev range is 0.594:0.594 km
mean CL zlev is 0.594 km
